# Leveled and Tiered Compaction, Measured

Companion notebook for the [LSM Trees and Compaction Strategies](https://codemia.io/courses/database_internals/dbi_lsm_trees_and_compaction_strategies) lesson on Codemia.

**What you will build**: a small LSM engine that really does flush, merge and rewrite its runs,
run under leveled and tiered compaction, and the three amplifications each strategy produces. By
the end you will be able to say what a fanout of 10 costs in writes, what it buys in reads, and why
the same database can be twice its live size and still be working correctly.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)

MEM = 2000       # entries the memtable holds before it flushes a run
WRITES = 300000  # writes in the workload
KEYS = 60000     # distinct keys, so most of those writes are updates

## Concept recap

An LSM tree never updates anything in place. Writes land in a memtable, the memtable is flushed
whole as an immutable sorted run, and runs are merged in the background. Everything interesting
follows from how that merging is organised.

**Leveled** keeps one run per level, each level a fixed multiple larger than the one above. A run
that overflows its level is merged into the level below, which means it is rewritten against data
that is already there. Each byte is therefore rewritten roughly once per level, times the fanout.
Reads are cheap because there is one run per level to consider.

**Tiered** lets runs accumulate at each level and merges them only when enough have piled up. Each
byte is written once per level and no more, so writing is much cheaper. Reads are not: a lookup may
have to consider every run at every level.

Three costs are worth separating. Write amplification is bytes written to storage per byte the
application wrote. Read amplification is runs consulted per lookup. Space amplification is bytes
stored per live byte, and it exists because an overwritten key is still sitting in an older run
until a merge discards it.

## An engine that flushes and merges

In [ ]:
class LSM:
    """Entry counting rather than byte counting, which keeps the arithmetic readable.

    A run is the set of keys it holds. Merging unions them and discards the older copy of
    any key present twice, which is exactly where space amplification goes away.
    """

    def __init__(self, strategy="leveled", fanout=4):
        self.strategy = strategy
        self.T = fanout
        self.levels = [[]]       # levels[i] is a list of runs, a run is a set of keys
        self.memtable = set()
        self.written = 0         # entries written to storage, by flushes and by merges
        self.inserted = 0
        self.trace = []          # (inserted, written) after every flush

    def cap(self, i):
        """Capacity of level i for the leveled strategy, in entries."""
        return MEM * self.T ** i

    def _level(self, i):
        while len(self.levels) <= i:
            self.levels.append([])
        return self.levels[i]

    def put(self, key):
        self.memtable.add(int(key))
        self.inserted += 1
        if len(self.memtable) >= MEM:
            self.flush()

    def flush(self):
        run, self.memtable = self.memtable, set()
        self.written += len(run)
        self._level(0).append(run)
        self.compact()
        self.trace.append((self.inserted, self.written))

    def merge(self, runs, into):
        self.written += sum(len(r) for r in runs)
        out = set()
        for r in runs:
            out |= r
        self._level(into).append(out)

    def compact(self):
        if self.strategy == "tiered":
            i = 0
            while i < len(self.levels):
                if len(self.levels[i]) >= self.T:
                    runs, self.levels[i] = self.levels[i], []
                    self.merge(runs, i + 1)
                i += 1
            return
        # leveled: level 0 stages runs, every level below holds exactly one
        if len(self.levels[0]) >= self.T:
            runs = self.levels[0] + self._level(1)
            self.levels[0], self.levels[1] = [], []
            self.merge(runs, 1)
        i = 1
        while i < len(self.levels):
            if self.levels[i] and len(self.levels[i][0]) > self.cap(i):
                runs = self.levels[i] + self._level(i + 1)
                self.levels[i], self.levels[i + 1] = [], []
                self.merge(runs, i + 1)
            i += 1

    def runs(self):
        return [r for level in self.levels for r in level]

    def stored(self):
        return sum(len(r) for r in self.runs())

    def live(self):
        out = set(self.memtable)
        for r in self.runs():
            out |= r
        return len(out)

    def report(self):
        runs = self.runs()
        return {
            "strategy": self.strategy,
            "fanout": self.T,
            "levels": sum(1 for lv in self.levels if lv),
            "runs": len(runs),
            "write_amp": self.written / self.inserted,
            "space_amp": self.stored() / self.live(),
        }

In [ ]:
def run_workload(strategy, fanout, n=WRITES, keys=KEYS, seed=11):
    gen = np.random.default_rng(seed)
    db = LSM(strategy=strategy, fanout=fanout)
    for k in gen.integers(0, keys, size=n):
        db.put(k)
    return db


leveled = run_workload("leveled", 4)
tiered = run_workload("tiered", 4)

for db in (leveled, tiered):
    r = db.report()
    print("%-8s fanout %d  levels %d  runs %2d  write amp %5.2f  space amp %5.2f" %
          (r["strategy"], r["fanout"], r["levels"], r["runs"], r["write_amp"], r["space_amp"]))

Same workload, same fanout, same data. Leveled writes each entry several times over and keeps the
run count down. Tiered writes substantially less and leaves more runs behind, and it holds on to far
more dead versions, which is the space amplification column. The gap in runs widens with the fanout:
in the sweep further down, a fanout of 16 leaves tiered with three times as many runs as leveled.

Neither number is a defect. They are the two ends of a single trade, and an engine that offers both
is offering you the choice rather than a better algorithm.

## Where the writes go, over time

In [ ]:
plt.figure(figsize=(9, 4))
for db, color in [(leveled, "#5b8def"), (tiered, "#d1495b")]:
    t = np.array(db.trace, dtype=float)
    plt.plot(t[:, 0] / 1000, t[:, 1] / 1000, color=color, lw=2, label=db.strategy)
plt.plot([0, WRITES / 1000], [0, WRITES / 1000], color="#8a8f98", ls="--", lw=1,
         label="what the application wrote")
plt.xlabel("thousands of entries written by the application")
plt.ylabel("thousands of entries written to storage")
plt.legend()
plt.title("cumulative write amplification")
plt.tight_layout()
plt.show()

The leveled line climbs steadily and the tiered line climbs in steps, and the steps are the point.
A tiered merge is rare and large: nothing happens for a long time and then several levels of runs
are rewritten at once. That burst is why tiered engines are associated with latency spikes and with
needing free space for a merge that has not started yet.

The leveled line is steeper overall and much smoother. It is more total work, spread evenly, which
is usually easier to live with than less work arriving all at once.

## What a lookup costs, with and without bloom filters

In [ ]:
def lookup_cost(db, fpr=0.01):
    """Runs consulted for a key that is not present, with and without a filter per run."""
    n = len(db.runs())
    return n, n * fpr


print("%-8s %6s %10s %16s" % ("strategy", "runs", "no filter", "with 1% filters"))
for db in (leveled, tiered):
    n, filtered = lookup_cost(db)
    print("%-8s %6d %10d %16.2f" % (db.strategy, n, n, filtered))

print()
print("a filter turns a lookup for an absent key from one read per run into almost none,")
print("and does nothing at all for a range scan, which has to open every run regardless")

This is the line that decides whether tiered compaction is usable. A bloom filter per run makes the
run count nearly irrelevant for point lookups that miss, which is the common case in a cache or a
key value store. It does nothing for a scan, because a scan cannot ask a filter about a range.

So the honest version of the trade is narrower than it first appears: tiered compaction costs reads
only when the reads are scans, or when the key is present and the filters cannot rule anything out.

## Variations: sweeping the fanout

In [ ]:
print("%-8s %7s %7s %6s %11s %11s" %
      ("strategy", "fanout", "levels", "runs", "write amp", "space amp"))
sweep = {}
for strategy in ("leveled", "tiered"):
    rows = []
    for T in (2, 4, 8, 16):
        r = run_workload(strategy, T).report()
        rows.append(r)
        print("%-8s %7d %7d %6d %11.2f %11.2f" %
              (r["strategy"], r["fanout"], r["levels"], r["runs"], r["write_amp"], r["space_amp"]))
    sweep[strategy] = rows

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for strategy, color in [("leveled", "#5b8def"), ("tiered", "#d1495b")]:
    rows = sweep[strategy]
    T = [r["fanout"] for r in rows]
    axes[0].plot(T, [r["write_amp"] for r in rows], "o-", color=color, label=strategy)
    axes[1].plot(T, [r["runs"] for r in rows], "o-", color=color, label=strategy)
axes[0].set_xlabel("fanout"); axes[0].set_ylabel("write amplification")
axes[1].set_xlabel("fanout"); axes[1].set_ylabel("runs a lookup may consult")
for ax in axes:
    ax.set_xscale("log", base=2)
    ax.legend()
plt.tight_layout()
plt.show()

Read the two panels together, because separately each one argues for the opposite setting.

A larger fanout means fewer levels, which means fewer rewrites per byte under leveled compaction and
fewer runs to consult. It also means each merge is larger, and under tiered compaction it means more
runs piled up at each level before anything is merged.

The settings people actually ship, a fanout around 10 with leveled compaction below the first level
or two, sit where these curves cross for a workload with more reads than writes. A write heavy
ingest pipeline moves the other way, and the engines that let you mix the two per level exist
because the right answer is often different at the top of the tree than at the bottom.

## What to try next

- Make the workload skewed rather than uniform, with ten percent of the keys taking ninety percent
  of the writes, and watch space amplification rise because the same hot keys sit in every run.
- Add deletes as tombstones that only disappear at the bottom level, and measure how long a deleted
  key stays readable in storage after the application stopped referring to it.
- Count the bytes a merge reads as well as the bytes it writes, which is what the storage device
  actually sees, and recompute the amplification against the device rather than the application.
- Model a tiered engine that keeps the last level leveled, which is what most production engines do,
  and see how much of the space amplification that one change removes.